In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')

# --- 1. LOAD DATA ---

pokemon_df  = pd.read_csv('pokemon.csv')
combats_df  = pd.read_csv('combats.csv')

print(f"Pokemon: {pokemon_df.shape} | Combats: {combats_df.shape}")

# Fix one known missing name
pokemon_df.loc[pokemon_df['#'] == 62, 'Name'] = 'Primeape'
pokemon_df['Type 2'] = pokemon_df['Type 2'].fillna('None')

# --- 2. COMPUTE WIN RATE PER POKEMON ---

# Count wins and total battles using vectorized operations (fast)
wins  = combats_df.groupby('Winner').size().rename('wins')
first = combats_df.groupby('First_pokemon').size().rename('as_first')
second = combats_df.groupby('Second_pokemon').size().rename('as_second')

totals = first.add(second, fill_value=0)
win_rate = (wins / totals).rename('Win_Rate')

pokemon_df['Win_Rate'] = pokemon_df['#'].map(win_rate)

print(f"Win rate computed for {win_rate.notna().sum()} Pokemon")
print(pokemon_df[['#', 'Name', 'Win_Rate']].head())

# --- 3. EXPLORATORY ANALYSIS ---

stat_cols = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Win_Rate']

plt.figure(figsize=(8, 6))
sns.heatmap(pokemon_df[stat_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Between Stats and Win Rate')
plt.tight_layout()
plt.show()

# Categorize win rate for color-coding
pokemon_df['Win_Rate_Cat'] = pd.cut(
    pokemon_df['Win_Rate'],
    bins=[0, 0.3, 0.5, 0.7, 1.0],
    labels=['Low', 'Medium', 'Good', 'Excellent']
)

g = sns.PairGrid(pokemon_df, vars=['HP', 'Attack', 'Defense', 'Speed'],
                 hue='Win_Rate_Cat', palette='viridis')
g.map_diag(sns.histplot, kde=True)
g.map_offdiag(sns.scatterplot, alpha=0.6)
g.add_legend(title='Win Rate')
plt.suptitle('Stats vs Win Rate', y=1.02)
plt.tight_layout()
plt.show()

print("\nTop 10 by win rate:")
print(pokemon_df.nlargest(10, 'Win_Rate')[['#', 'Name', 'Win_Rate', 'HP', 'Attack', 'Speed']].to_string(index=False))

print("\nBottom 10 by win rate:")
print(pokemon_df.nsmallest(10, 'Win_Rate')[['#', 'Name', 'Win_Rate', 'HP', 'Attack', 'Speed']].to_string(index=False))

# --- 4. PREPARE FEATURES FOR MACHINE LEARNING ---

# One-hot encode Type 1, Type 2, and Legendary
df_ml = pd.get_dummies(pokemon_df, columns=['Type 1', 'Type 2', 'Legendary'], drop_first=True)

drop_cols = ['#', 'Name', 'Win_Rate', 'Win_Rate_Cat']
feature_cols = [c for c in df_ml.columns if c not in drop_cols]

X = df_ml[feature_cols].fillna(0)
y = pokemon_df['Win_Rate']

# Only keep rows with a known win rate
mask = y.notna()
X, y = X[mask], y[mask]

print(f"Final shapes: X={X.shape}, y={y.shape}")

# --- 5. TRAIN / TEST SPLIT AND SCALE ---

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# --- 6. TRAIN AND COMPARE THREE MODELS ---

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost':           XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
}

mae_results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_pred)
    mae_results[name] = mae
    print(f"{name:<22} MAE: {mae:.4f}")

plt.figure(figsize=(8, 5))
bars = plt.bar(mae_results.keys(), mae_results.values(), color=['blue', 'green', 'red'])
plt.ylabel('Mean Absolute Error (MAE)')
plt.title('Model Comparison — Pokemon Win Rate Prediction')
for bar, val in zip(bars, mae_results.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{val:.4f}', ha='center', va='bottom')
plt.ylim(0, max(mae_results.values()) * 1.15)
plt.tight_layout()
plt.show()

# --- 7. PCA — VISUALIZE POKEMON IN 2D ---

pca = PCA()
X_pca = pca.fit_transform(X_train_scaled)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(range(1, len(explained) + 1), explained, alpha=0.7)
axes[0].set_title('Variance per Component')
axes[0].set_xlabel('Component')
axes[0].set_ylabel('Explained Variance')

axes[1].plot(range(1, len(cumulative) + 1), cumulative, 'bo-', markersize=4)
axes[1].axhline(0.95, color='r', linestyle='--', label='95%')
axes[1].set_title('Cumulative Variance')
axes[1].set_xlabel('Number of Components')
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"First 2 components explain: {cumulative[1]:.2%} of variance")

plt.figure(figsize=(10, 6))
sc = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_train, cmap='viridis', alpha=0.7, edgecolors='k')
plt.colorbar(sc, label='Win Rate')
plt.xlabel(f'PC1 ({explained[0]:.2%} variance)')
plt.ylabel(f'PC2 ({explained[1]:.2%} variance)')
plt.title('PCA Projection of Pokemon by Win Rate')
plt.tight_layout()
plt.show()